In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-10 22:00:30--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.213, 18.239.115.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  11.7MB/s    in 5.6s    

2026-03-10 22:00:36 (12.0 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 22:01:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet",)


In [9]:
df = df.repartition(4)

In [16]:
df.write.mode("overwrite").parquet("data/hw/")

In [13]:
ls

covert to PQ.ipynb               Spark SQL.ipynb
data/                            spark-warehouse/
fhvhv/                           taxi_zone_lookup.csv
fhvhv_tripdata_2021-01.csv       Test Spark.ipynb
head.csv                         Untitled.ipynb
notes                            wc
pyspark.ipynb                    yellow_tripdata_2025-11.parquet
Spark Homework.ipynb             zones/


In [18]:
spark.version

'4.1.1'

In [19]:
df.columns


['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [23]:
df.createOrReplaceTempView("trips")

In [24]:
spark.sql("""
    SELECT COUNT(*) AS trip_count
    FROM trips
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+----------+
|trip_count|
+----------+
|    162604|
+----------+



In [25]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-10 22:19:11--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.86, 18.239.115.213, 18.239.115.146, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 22:19:11 (5.74 GB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [26]:
import pandas as pd

In [27]:
df_zones = pd.read_csv("taxi_zone_lookup.csv")

In [30]:
spark_zones  = spark.createDataFrame(df_zones)

In [32]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [31]:
spark_zones.printSchema()

root
 |-- locationid: long (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [33]:
df_final = df.join(spark_zones,df.PULocationID == spark_zones.locationid)

In [34]:
df_final.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- locationid: long (nullable = true)
 |-- borough: string (nullable = tru

In [39]:
df_final.groupBy("zone").count().orderBy("count").show()  

+--------------------+-----+
|                zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|        Crotona Park|   14|
|New Dorp/Midland ...|   14|
|             Oakwood|   14|
|       West Brighton|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows


In [40]:
df_final.createOrReplaceTempView("trips_final")

In [42]:
from pyspark.sql import functions as F



In [46]:
df_final.select(
    (
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
    ).alias("trip_hours").max()
)

TypeError: 'Column' object is not callable

In [47]:

df_final.select(
    ((
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 3600
).alias("trip_hours")).select(
    F.max("trip_hours").alias("longest_trip_hours")
).show(truncate=False)

+------------------+
|longest_trip_hours|
+------------------+
|90.64666666666666 |
+------------------+

